# Smart Waste Collection Agent — IDA* Search

**Course:** AIMLCZG557 / AECLZG557 — Artificial and Computational Intelligence  
**Assignment:** Assignment 1 — PS1 (Waste Collection)  
**Semester:** S2 2025-2026  
**Group ID:** G225  

---

## Problem Statement

The Municipal Corporation of Bengaluru has deployed a Smart Waste Collection Robot Agent. The city's waste management network is modelled as a **weighted undirected graph** where:

- **Vertices** — waste collection centres or processing units
- **Edges** — roads connecting those locations
- **Edge weights** — travel cost (fuel / distance)

The robot must find the **least-cost route** from a source location to a destination using the **IDA\*** (Iterative Deepening A\*) algorithm.

---

## 1. PEAS Analysis

| Component | Description |
|---|---|
| **Performance Measure** | Minimise total travel cost (fuel / distance) from source to destination; secondary goal: minimise number of nodes explored |
| **Environment** | Bengaluru city road network modelled as a static, fully observable, deterministic, weighted undirected graph |
| **Actuators** | Robot movement engine — traverses edges between nodes; can move to any directly connected neighbour |
| **Sensors** | GPS / location sensor (current node identity); road-cost sensor (edge weights); destination sensor (goal node identity) |

**Environment properties:**
- **Fully observable** — the complete road network is known before search begins
- **Deterministic** — traversing an edge always costs the stated weight
- **Static** — the graph does not change during the robot's journey
- **Discrete** — finite number of nodes and edges

---

## 2. Heuristic Choice and Justification

### Heuristic Function — h(n)

Since no geographical coordinates are available, the heuristic for each node is defined as a **manually computed lower-bound estimate** of the travel cost from that node to the goal. These values are supplied in the input file alongside the graph, making the approach fully data-driven.

**Admissibility:** Every heuristic value `h(n)` is less than or equal to the true shortest-path cost from `n` to the goal.  
**Consistency (monotonicity):** For every edge `(n, m, w)`, `h(n) ≤ w + h(m)`, so the f-values are non-decreasing along any path, guaranteeing IDA* will not re-explore worse paths.

### How values were derived

For each node, the actual shortest-path cost to the destination was computed (by inspection / Dijkstra), and the heuristic value was set to be equal to or slightly less than that value — never exceeding it.

**Example (Case 1, Destination: Yelahanka):**

| Node | Actual Shortest Cost | h(n) |
|---|---|---|
| MG_Road | 8 | 6 |
| Electronic_City | 6 | 4 |
| Whitefield | 4 | 4 |
| Koramangala | 8 | 6 |
| Jayanagar | 2 | 2 |
| Hebbal | 5 | 4 |
| Yelahanka | 0 | 0 |

All values satisfy `h(n) ≤ actual cost` → heuristic is **admissible**.  
The non-zero values provide meaningful guidance, pruning many branches and reducing the number of IDA* iterations compared to a zero heuristic.

---

## 3. Algorithm — IDA* and Cost Function

### Cost Function — f(n)

$$f(n) = g(n) + h(n)$$

- `g(n)` — actual cost accumulated from the source node to node `n` along the current path
- `h(n)` — admissible heuristic estimate of the remaining cost from `n` to the goal

A node is **pruned** when `f(n) > threshold`.

### IDA* Algorithm

IDA* combines the memory efficiency of **Iterative Deepening Depth-First Search (IDDFS)** with the informed guidance of **A\***:

1. Initialise `threshold = h(start)`
2. Run a **depth-first search** from the start node, pruning any node where `f(n) > threshold`
3. If the goal is reached → **return** the path and cost
4. If the goal is not reached → set `threshold = min f-value that exceeded the current threshold`
5. Repeat from step 2 with the new threshold

### Why IDA* for this problem?

| Property | Justification |
|---|---|
| **Optimal** | With an admissible heuristic, IDA* is guaranteed to find the least-cost path |
| **Memory efficient** | Uses O(d) space (depth of solution), unlike A* which stores all open nodes |
| **Informed** | The heuristic prunes large portions of the search space, unlike uninformed BFS/DFS |
| **Complete** | Will always find a solution if one exists (finite, positive-weight graph) |

An alternative would be **Dijkstra's algorithm** (no heuristic), but it explores more nodes and has higher memory usage. **A\*** is also optimal but requires O(b^d) memory. **IDA\*** balances both.

---

## 4. Implementation

In [1]:
# ── Imports ──────────────────────────────────────────────────────────────────
import sys
import os

In [2]:
# ── Input Parsing ─────────────────────────────────────────────────────────────

def parse_input(filename):
    """
    Reads the input file and returns a list of case dictionaries.
    Each case dictionary has keys:
        'case_num'    : int
        'num_nodes'   : int
        'num_edges'   : int
        'edges'       : list of (node1, node2, weight)
        'heuristic'   : dict {node: h_value}
        'source'      : str
        'destination' : str
    """
    if not os.path.isfile(filename):
        raise FileNotFoundError(f"Input file '{filename}' not found.")

    cases = []
    current = None

    with open(filename, 'r') as fh:
        for raw_line in fh:
            line = raw_line.strip()
            if not line:          # blank separator between cases
                continue

            tokens = line.split()
            keyword = tokens[0].upper()

            if keyword == 'CASE':
                if current is not None:
                    cases.append(current)
                current = {
                    'case_num'    : int(tokens[1]),
                    'num_nodes'   : 0,
                    'num_edges'   : 0,
                    'edges'       : [],
                    'heuristic'   : {},
                    'source'      : None,
                    'destination' : None
                }
            elif keyword == 'NODES':
                current['num_nodes'] = int(tokens[1])
            elif keyword == 'EDGES':
                current['num_edges'] = int(tokens[1])
            elif keyword == 'HEURISTIC':
                node, h_val = tokens[1], float(tokens[2])
                current['heuristic'][node] = h_val
            elif keyword == 'SOURCE':
                current['source'] = tokens[1]
            elif keyword == 'DESTINATION':
                current['destination'] = tokens[1]
            else:
                # Edge line: node1 node2 weight
                if len(tokens) == 3:
                    try:
                        n1, n2, w = tokens[0], tokens[1], float(tokens[2])
                        current['edges'].append((n1, n2, w))
                    except ValueError:
                        raise ValueError(f"Malformed edge line: '{line}'")

    if current is not None:
        cases.append(current)

    # ── Validation ────────────────────────────────────────────────────────────
    for case in cases:
        if case['source'] is None:
            raise ValueError(f"Case {case['case_num']}: SOURCE not specified.")
        if case['destination'] is None:
            raise ValueError(f"Case {case['case_num']}: DESTINATION not specified.")
        if not case['edges']:
            raise ValueError(f"Case {case['case_num']}: No edges found — graph is empty.")

    return cases

In [3]:
# ── Graph Builder ─────────────────────────────────────────────────────────────

def build_graph(edges):
    """
    Constructs an undirected weighted adjacency list from a list of
    (node1, node2, weight) tuples.
    Returns: dict  {node: [(neighbour, weight), ...]}
    """
    graph = {}
    for n1, n2, w in edges:
        graph.setdefault(n1, []).append((n2, w))
        graph.setdefault(n2, []).append((n1, w))
    return graph

In [4]:
# ── IDA* Core ────────────────────────────────────────────────────────────────

def _dfs(path, g, threshold, graph, goal, heuristic, counter):
    """
    Depth-first search subroutine for IDA*.

    Parameters
    ----------
    path      : list  — current path from source (mutable, used as stack)
    g         : float — accumulated cost from source to current node
    threshold : float — current f-value cutoff
    graph     : dict  — adjacency list
    goal      : str   — destination node name
    heuristic : dict  — h(n) values
    counter   : list  — single-element list [int] to track nodes explored
                        (mutable so recursive calls share the same counter)

    Returns
    -------
    'FOUND' if goal reached, else float (min f-value that exceeded threshold)
    """
    current = path[-1]
    f = g + heuristic.get(current, 0.0)

    if f > threshold:
        return f          # exceeded — report this f-value for next threshold

    if current == goal:
        return 'FOUND'

    minimum = float('inf')
    path_set = set(path)  # O(1) cycle check

    neighbours = graph.get(current, [])
    # Sort neighbours for deterministic, reproducible output
    for neighbour, edge_cost in sorted(neighbours, key=lambda x: x[0]):
        if neighbour in path_set:   # avoid cycles
            continue

        counter[0] += 1             # count every node pushed onto path
        path.append(neighbour)

        result = _dfs(path, g + edge_cost, threshold,
                      graph, goal, heuristic, counter)

        if result == 'FOUND':
            return 'FOUND'

        if result < minimum:
            minimum = result

        path.pop()

    return minimum


def ida_star(graph, source, destination, heuristic):
    """
    IDA* search from source to destination.

    Returns
    -------
    (path, total_cost, nodes_explored)
    Raises RuntimeError if no path exists.
    """
    if source not in graph:
        raise ValueError(f"Source node '{source}' not found in graph.")
    if destination not in graph:
        raise ValueError(f"Destination node '{destination}' not found in graph.")

    threshold = heuristic.get(source, 0.0)
    path = [source]
    counter = [1]          # source itself counts as the first explored node

    while True:
        result = _dfs(path, 0.0, threshold, graph, destination, heuristic, counter)

        if result == 'FOUND':
            total_cost = sum(
                next(w for nb, w in graph[path[i]] if nb == path[i + 1])
                for i in range(len(path) - 1)
            )
            return path, total_cost, counter[0]

        if result == float('inf'):
            raise RuntimeError(
                f"No path found from '{source}' to '{destination}'."
            )

        threshold = result   # raise threshold to smallest exceeded f-value

In [5]:
# ── Output Formatting and Writing ─────────────────────────────────────────────

def format_case_output(case_num, source, destination, path, total_cost, nodes_explored):
    """
    Formats the result of one IDA* case as a multi-line string.
    """
    path_str = ' -> '.join(path)
    sequence  = ', '.join(path)
    lines = [
        f"=== Case {case_num} ===",
        f"Source      : {source}",
        f"Destination : {destination}",
        f"Optimal Path: {path_str}",
        f"Total Travel Cost: {int(total_cost) if total_cost == int(total_cost) else total_cost}",
        f"Nodes Explored   : {nodes_explored}",
        f"Sequence of Locations: {sequence}",
        ""
    ]
    return '\n'.join(lines)


def write_output(filename, content):
    """
    Writes content string to the given output file.
    Raises IOError with a descriptive message on failure.
    """
    try:
        with open(filename, 'w') as fh:
            fh.write(content)
    except IOError as exc:
        raise IOError(f"Could not write to output file '{filename}': {exc}") from exc

In [6]:
# ── Main ──────────────────────────────────────────────────────────────────────

def main(input_file='inputPS1.txt', output_file='outputPS1.txt'):
    output_lines = []

    # ── Parse ─────────────────────────────────────────────────────────────────
    try:
        cases = parse_input(input_file)
    except (FileNotFoundError, ValueError) as exc:
        print(f"[ERROR] {exc}")
        return

    # ── Process each case ─────────────────────────────────────────────────────
    for case in cases:
        case_num    = case['case_num']
        source      = case['source']
        destination = case['destination']
        heuristic   = case['heuristic']

        graph = build_graph(case['edges'])

        # Ensure all nodes reachable from source have a heuristic value
        for node in graph:
            if node not in heuristic:
                print(f"[WARNING] Case {case_num}: No heuristic for node '{node}'. Defaulting h=0.")
                heuristic[node] = 0.0

        print(f"Running IDA* for Case {case_num}: {source} → {destination}")

        try:
            path, total_cost, nodes_explored = ida_star(graph, source, destination, heuristic)
            block = format_case_output(
                case_num, source, destination, path, total_cost, nodes_explored
            )
        except (ValueError, RuntimeError) as exc:
            block = (
                f"=== Case {case_num} ===\n"
                f"Source      : {source}\n"
                f"Destination : {destination}\n"
                f"[ERROR] {exc}\n"
            )

        output_lines.append(block)
        print(block)

    # ── Write output file ─────────────────────────────────────────────────────
    full_output = '\n'.join(output_lines)
    try:
        write_output(output_file, full_output)
        print(f"Output written to '{output_file}'.")
    except IOError as exc:
        print(f"[ERROR] {exc}")


# ── Entry point ───────────────────────────────────────────────────────────────
main()

Running IDA* for Case 1: MG_Road → Yelahanka
=== Case 1 ===
Source      : MG_Road
Destination : Yelahanka
Optimal Path: MG_Road -> Electronic_City -> Whitefield -> Yelahanka
Total Travel Cost: 8
Nodes Explored   : 8
Sequence of Locations: MG_Road, Electronic_City, Whitefield, Yelahanka

Running IDA* for Case 2: A → E
=== Case 2 ===
Source      : A
Destination : E
Optimal Path: A -> C -> D -> E
Total Travel Cost: 5
Nodes Explored   : 9
Sequence of Locations: A, C, D, E

Output written to 'outputPS1.txt'.


In [7]:
# ── Display outputPS1.txt ─────────────────────────────────────────────────────
with open('outputPS1.txt', 'r') as fh:
    print(fh.read())

=== Case 1 ===
Source      : MG_Road
Destination : Yelahanka
Optimal Path: MG_Road -> Electronic_City -> Whitefield -> Yelahanka
Total Travel Cost: 8
Nodes Explored   : 8
Sequence of Locations: MG_Road, Electronic_City, Whitefield, Yelahanka

=== Case 2 ===
Source      : A
Destination : E
Optimal Path: A -> C -> D -> E
Total Travel Cost: 5
Nodes Explored   : 9
Sequence of Locations: A, C, D, E

